# Global Top-11 Product Baseline on Test

This notebook keeps the same split and metric definitions as `final_experiments.ipynb`, but uses a simple baseline:

- build the global `top11` products from the **train users' history**
- predict those same 11 products for every test user
- evaluate on the **test candidate universe** with `Test AUC`, `Test PR-AUC`, and `F1@11`

It first tries to load the leakage-safe base bundle. If that bundle is missing, it falls back to rebuilding the minimal required tables from the raw Instacart CSVs using the same leakage-safe definitions as `leakage_safe_preprocess`.


In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score

try:
    import kagglehub
except Exception:
    kagglehub = None

RANDOM_STATE = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TOP_K_FIXED = 11
DEFAULT_DATASET = 'yasserh/instacart-online-grocery-basket-analysis-dataset'


def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


cwd = Path.cwd().resolve()
bundle_candidates = [
    cwd / 'artifacts' / 'base_bundle',
    cwd.parent / 'artifacts' / 'base_bundle',
    cwd / 'archive' / 'Final version' / 'artifacts' / 'base_bundle',
    Path('/artifacts/base_bundle'),
]
raw_dir_candidates = [
    cwd / 'data',
    cwd / 'instacart_data',
    cwd.parent / 'data',
    cwd.parent / 'instacart_data',
]
output_candidates = [
    cwd / 'outputs',
    cwd / 'artifacts' / 'experiment_outputs',
    cwd.parent / 'artifacts' / 'experiment_outputs',
    Path('/artifacts/experiment_outputs'),
]

BUNDLE_DIR = first_existing_path(bundle_candidates)
RAW_DATA_DIR = first_existing_path(raw_dir_candidates)
OUTPUT_DIR = first_existing_path(output_candidates) or (cwd / 'outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    'BUNDLE_DIR': None if BUNDLE_DIR is None else str(BUNDLE_DIR),
    'RAW_DATA_DIR': None if RAW_DATA_DIR is None else str(RAW_DATA_DIR),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'HAS_KAGGLEHUB': kagglehub is not None,
})


{'BUNDLE_DIR': None, 'RAW_DATA_DIR': None, 'OUTPUT_DIR': '/Users/bettinabopeng/Documents/GitHub/5971/outputs', 'HAS_KAGGLEHUB': True}


In [3]:
def verify_required_files(data_dir: Path) -> None:
    required = [
        'orders.csv',
        'order_products__prior.csv',
        'order_products__train.csv',
        'products.csv',
        'aisles.csv',
        'departments.csv',
    ]
    missing = [name for name in required if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f'Missing required raw files under {data_dir}: {missing}')


def resolve_raw_data_dir(raw_data_dir: Path | None, dataset_name: str) -> Path:
    if raw_data_dir is not None and raw_data_dir.exists():
        verify_required_files(raw_data_dir)
        return raw_data_dir
    if kagglehub is None:
        raise FileNotFoundError(
            'Could not find a ready base bundle or local raw CSV folder, and `kagglehub` is unavailable.'
        )
    downloaded = Path(kagglehub.dataset_download(dataset_name))
    verify_required_files(downloaded)
    return downloaded


def is_literal_missing(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq('missing')


def read_raw_tables(data_dir: Path) -> dict:
    orders = pd.read_csv(data_dir / 'orders.csv')
    prior = pd.read_csv(data_dir / 'order_products__prior.csv')
    train = pd.read_csv(data_dir / 'order_products__train.csv')
    products = pd.read_csv(data_dir / 'products.csv')
    aisles = pd.read_csv(data_dir / 'aisles.csv')
    departments = pd.read_csv(data_dir / 'departments.csv')

    orders['days_since_prior_order'] = orders['days_since_prior_order'].fillna(0).astype(np.float32)
    orders = orders.sort_values(['user_id', 'order_number']).reset_index(drop=True)
    orders['user_cum_days'] = orders.groupby('user_id')['days_since_prior_order'].cumsum().astype(np.float32)

    return {
        'orders': orders,
        'prior': prior,
        'train': train,
        'products': products,
        'aisles': aisles,
        'departments': departments,
    }


def build_minimal_bundle_from_raw(tables: dict) -> dict:
    orders = tables['orders']
    prior = tables['prior']
    train = tables['train']
    products = tables['products']
    aisles = tables['aisles']
    departments = tables['departments']

    aisle_missing_mask = is_literal_missing(aisles['aisle'])
    department_missing_mask = is_literal_missing(departments['department'])
    product_missing_mask = is_literal_missing(products['product_name'])

    missing_aisle_ids = set(aisles.loc[aisle_missing_mask, 'aisle_id'])
    missing_department_ids = set(departments.loc[department_missing_mask, 'department_id'])
    invalid_product_mask = (
        product_missing_mask
        | products['aisle_id'].isin(missing_aisle_ids)
        | products['department_id'].isin(missing_department_ids)
    )

    valid_products = products.loc[~invalid_product_mask].copy()
    valid_product_ids = set(valid_products['product_id'])

    aisles = aisles.loc[~aisle_missing_mask].copy()
    departments = departments.loc[~department_missing_mask].copy()
    prior = prior[prior['product_id'].isin(valid_product_ids)].copy()
    train = train[train['product_id'].isin(valid_product_ids)].copy()

    product_meta = (
        valid_products.merge(aisles, on='aisle_id', how='left')
        .merge(departments, on='department_id', how='left')
        .rename(columns={'aisle': 'aisle_name', 'department': 'department_name'})
    )

    train_orders = orders.loc[orders['eval_set'] == 'train'].copy()
    test_orders = orders.loc[orders['eval_set'] == 'test'].copy()
    supervised_users = set(orders.loc[orders['eval_set'].isin(['train', 'test']), 'user_id'])
    prior_orders = orders.loc[(orders['eval_set'] == 'prior') & (orders['user_id'].isin(supervised_users))].copy()

    train_target_orders = train_orders[
        ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'user_cum_days']
    ].rename(
        columns={
            'order_id': 'target_order_id',
            'order_number': 'target_order_number',
            'order_dow': 'target_order_dow',
            'order_hour_of_day': 'target_order_hour_of_day',
            'days_since_prior_order': 'target_days_since_prior_order',
            'user_cum_days': 'target_user_cum_days',
        }
    )
    train_target_orders['target_source'] = 'train_order'

    test_target_orders = (
        prior_orders.loc[prior_orders['user_id'].isin(test_orders['user_id'])]
        .sort_values(['user_id', 'order_number'])
        .groupby('user_id')
        .tail(1)[
            ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'user_cum_days']
        ]
        .rename(
            columns={
                'order_id': 'target_order_id',
                'order_number': 'target_order_number',
                'order_dow': 'target_order_dow',
                'order_hour_of_day': 'target_order_hour_of_day',
                'days_since_prior_order': 'target_days_since_prior_order',
                'user_cum_days': 'target_user_cum_days',
            }
        )
    )
    test_target_orders['target_source'] = 'prior_as_target'

    target_orders = (
        pd.concat([train_target_orders, test_target_orders], ignore_index=True)
        .sort_values('user_id')
        .reset_index(drop=True)
    )

    prior_detail_full = (
        prior.merge(
            prior_orders[
                ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'user_cum_days']
            ],
            on='order_id',
            how='left',
        )
        .merge(product_meta, on='product_id', how='left')
        .sort_values(['user_id', 'order_number', 'add_to_cart_order', 'product_id'])
        .reset_index(drop=True)
    )

    train_target_detail = (
        train.merge(
            train_target_orders[['target_order_id', 'user_id', 'target_order_number', 'target_source']],
            left_on='order_id',
            right_on='target_order_id',
            how='left',
        )
        .merge(product_meta, on='product_id', how='left')
        .sort_values(['user_id', 'target_order_number', 'add_to_cart_order', 'product_id'])
        .reset_index(drop=True)
    )

    test_target_detail = (
        prior_detail_full.merge(
            test_target_orders[['target_order_id', 'user_id', 'target_order_number', 'target_source']],
            left_on=['order_id', 'user_id'],
            right_on=['target_order_id', 'user_id'],
            how='inner',
        )
        .drop(columns=['order_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'user_cum_days'])
        .sort_values(['user_id', 'target_order_number', 'add_to_cart_order', 'product_id'])
        .reset_index(drop=True)
    )

    target_detail = (
        pd.concat([train_target_detail, test_target_detail], ignore_index=True)
        .sort_values(['user_id', 'target_order_number', 'add_to_cart_order', 'product_id'])
        .reset_index(drop=True)
    )

    history_detail = (
        prior_detail_full.merge(
            target_orders[['user_id', 'target_order_number']],
            on='user_id',
            how='inner',
        )
        .query('order_number < target_order_number')
        .drop(columns=['target_order_number'])
        .sort_values(['user_id', 'order_number', 'add_to_cart_order', 'product_id'])
        .reset_index(drop=True)
    )

    labels = (
        target_detail[['user_id', 'product_id']]
        .drop_duplicates()
        .assign(label=np.int8(1))
        .reset_index(drop=True)
    )

    candidate_universe = (
        history_detail[['user_id', 'product_id']]
        .drop_duplicates()
        .sort_values(['user_id', 'product_id'])
        .reset_index(drop=True)
    )

    return {
        'product_meta': product_meta,
        'history_detail': history_detail,
        'target_orders': target_orders,
        'labels': labels,
        'candidate_universe': candidate_universe,
    }


def load_or_build_bundle() -> dict:
    if BUNDLE_DIR is not None and (BUNDLE_DIR / 'bundle_metadata.json').exists():
        print(f'Loading existing base bundle from: {BUNDLE_DIR}')
        return {
            'product_meta': pd.read_parquet(BUNDLE_DIR / 'product_meta.parquet'),
            'history_detail': pd.read_parquet(BUNDLE_DIR / 'history_detail.parquet'),
            'target_orders': pd.read_parquet(BUNDLE_DIR / 'target_orders.parquet'),
            'labels': pd.read_parquet(BUNDLE_DIR / 'labels.parquet'),
            'candidate_universe': pd.read_parquet(BUNDLE_DIR / 'candidate_universe.parquet'),
        }

    resolved_raw_dir = resolve_raw_data_dir(RAW_DATA_DIR, DEFAULT_DATASET)
    print(f'Building minimal bundle from raw data: {resolved_raw_dir}')
    tables = read_raw_tables(resolved_raw_dir)
    return build_minimal_bundle_from_raw(tables)


In [4]:
bundle = load_or_build_bundle()

all_users = np.sort(bundle['target_orders']['user_id'].unique())
rng = np.random.default_rng(RANDOM_STATE)
shuffled_users = rng.permutation(all_users)

n_total = len(shuffled_users)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VALIDATION_RATIO)
n_test = n_total - n_train - n_val

if min(n_train, n_val, n_test) <= 0:
    raise ValueError('Train / validation / test split is invalid.')

train_users = np.sort(shuffled_users[:n_train])
val_users = np.sort(shuffled_users[n_train:n_train + n_val])
test_users = np.sort(shuffled_users[n_train + n_val:])

split_overview = pd.DataFrame([
    {'split': 'train', 'users': len(train_users)},
    {'split': 'validation', 'users': len(val_users)},
    {'split': 'test', 'users': len(test_users)},
])
display(split_overview)

print({
    'history_rows': int(len(bundle['history_detail'])),
    'candidate_rows': int(len(bundle['candidate_universe'])),
    'positive_pairs': int(len(bundle['labels'])),
})


Building minimal bundle from raw data: /Users/bettinabopeng/.cache/kagglehub/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset/versions/1


,split,users
0,train,144346
1,validation,30931
2,test,30932


{'history_rows': 31588436, 'candidate_rows': 12945247, 'positive_pairs': 2153274}


In [5]:
def build_candidate_frame_for_users(user_ids: np.ndarray) -> pd.DataFrame:
    frame = (
        bundle['candidate_universe'][bundle['candidate_universe']['user_id'].isin(user_ids)]
        .merge(bundle['target_orders'][['user_id', 'target_order_number']], on='user_id', how='left')
        .merge(bundle['labels'], on=['user_id', 'product_id'], how='left')
        .merge(
            bundle['product_meta'][['product_id', 'product_name', 'aisle_name', 'department_name']],
            on='product_id',
            how='left',
        )
        .copy()
    )
    frame['label'] = frame['label'].fillna(0).astype(int)
    frame['u_avg_basket_size'] = float(TOP_K_FIXED)
    return frame


def eval_row_level(y_true, y_prob):
    y_prob = np.asarray(y_prob, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    y_prob_clip = np.clip(y_prob, 1e-15, 1 - 1e-15)
    return {
        'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc': average_precision_score(y_true, y_prob),
        'logloss': log_loss(y_true, y_prob_clip),
    }


def make_k_map(df_input: pd.DataFrame) -> dict:
    user_k = df_input.groupby('user_id')['u_avg_basket_size'].first().reset_index()
    user_k['pred_k'] = TOP_K_FIXED
    return dict(zip(user_k['user_id'], user_k['pred_k']))


def eval_order_with_k_map(df_eval: pd.DataFrame, k_map: dict, prefix: str):
    ranked = df_eval.sort_values(['user_id', 'prob', 'product_id'], ascending=[True, False, True]).copy()
    ranked['rank_within_user'] = ranked.groupby('user_id').cumcount() + 1
    ranked['pred_k'] = ranked['user_id'].map(k_map).fillna(TOP_K_FIXED).astype(int).clip(lower=1)
    ranked['pred_topk'] = (ranked['rank_within_user'] <= ranked['pred_k']).astype(int)

    precision_vals, recall_vals, f1_vals, hit_vals, avgk_vals = [], [], [], [], []
    for _, g in ranked.groupby('user_id'):
        y_t = g['label'].to_numpy()
        y_p = g['pred_topk'].to_numpy()
        precision_vals.append(precision_score(y_t, y_p, zero_division=0))
        recall_vals.append(recall_score(y_t, y_p, zero_division=0))
        f1_vals.append(f1_score(y_t, y_p, zero_division=0))
        hit_vals.append(float((g.loc[g['pred_topk'] == 1, 'label'].sum() > 0)))
        avgk_vals.append(float(g['pred_k'].iloc[0]))

    return {
        f'precision@{prefix}': float(np.mean(precision_vals)),
        f'recall@{prefix}': float(np.mean(recall_vals)),
        f'f1@{prefix}': float(np.mean(f1_vals)),
        f'hit@{prefix}': float(np.mean(hit_vals)),
        f'avg_k@{prefix}': float(np.mean(avgk_vals)),
    }


In [6]:
train_history = bundle['history_detail'][bundle['history_detail']['user_id'].isin(train_users)].copy()

global_top11 = (
    train_history.groupby(['product_id', 'product_name'], dropna=False)
    .size()
    .reset_index(name='train_purchase_count')
    .sort_values(['train_purchase_count', 'product_id'], ascending=[False, True])
    .head(TOP_K_FIXED)
    .reset_index(drop=True)
)

top11_product_ids = set(global_top11['product_id'].tolist())
display(global_top11)


,product_id,product_name,train_purchase_count
0,24852,Banana,321252
1,13176,Bag of Organic Bananas,258222
2,21137,Organic Strawberries,181168
3,21903,Organic Baby Spinach,165030
4,47209,Organic Hass Avocado,145470
5,47766,Organic Avocado,121532
6,47626,Large Lemon,102990
7,16797,Strawberries,96884
8,26209,Limes,96410
9,27845,Organic Whole Milk,94329


In [7]:
test_df = build_candidate_frame_for_users(test_users)
test_df['prob'] = test_df['product_id'].isin(top11_product_ids).astype(float)

metrics = {
    'model': 'GlobalTop11FromTrain',
    'split': 'test',
    'n_test_users': int(test_df['user_id'].nunique()),
    'n_test_rows': int(len(test_df)),
}
metrics.update(eval_row_level(test_df['label'].to_numpy(), test_df['prob'].to_numpy()))
metrics.update(
    eval_order_with_k_map(
        test_df[['user_id', 'product_id', 'label', 'u_avg_basket_size', 'prob']].copy(),
        make_k_map(test_df),
        '11',
    )
)

result_df = pd.DataFrame([metrics])
display(result_df[[
    'model', 'split', 'auc', 'pr_auc', 'f1@11', 'precision@11', 'recall@11', 'hit@11', 'avg_k@11'
]])

result_path = OUTPUT_DIR / 'global_top11_test_baseline_metrics.csv'
pred_path = OUTPUT_DIR / 'global_top11_test_predictions.parquet'
top11_path = OUTPUT_DIR / 'global_top11_products.csv'

result_df.to_csv(result_path, index=False)
test_df.to_parquet(pred_path, index=False)
global_top11.to_csv(top11_path, index=False)

print({
    'metrics_csv': str(result_path),
    'predictions_parquet': str(pred_path),
    'top11_products_csv': str(top11_path),
})


,model,split,auc,pr_auc,f1@11,precision@11,recall@11,hit@11,avg_k@11
0,GlobalTop11FromTrain,test,0.533343,0.113623,0.195065,0.162118,0.34813,0.746088,11.0


{'metrics_csv': '/Users/bettinabopeng/Documents/GitHub/5971/outputs/global_top11_test_baseline_metrics.csv', 'predictions_parquet': '/Users/bettinabopeng/Documents/GitHub/5971/outputs/global_top11_test_predictions.parquet', 'top11_products_csv': '/Users/bettinabopeng/Documents/GitHub/5971/outputs/global_top11_products.csv'}
